In [7]:
import pandas as pd
from scipy import stats
import spacy

In [ ]:
nlp_it = spacy.load("it_core_news_lg")

## Spearman corr between MAS-IT and valence math

Correlation between each participant's total MAS-IT score and the valence of *math*, for the four
groups at once. Two measures of "valence of math" are computed:
- **mean_of_associations** — mean valence of the participant's 3 associations to *matematica*
- **math_word_only** — valence of the cue word *matematica* itself (`valence_cue_word`)

In [8]:
associations = pd.read_csv(
    "/Users/emma/projects/cognoscolab/rassereno/out/formatting/database/all_humans_associations_valence.csv"
)
mas_it = pd.read_csv(
    "/Users/emma/projects/cognoscolab/rassereno/out/formatting/database/all_humans_mas_it.csv"
)

associations_gpt = pd.read_csv(
    "/Users/emma/projects/cognoscolab/rassereno/out/formatting/database/all_gpt_associations_valence.csv"
)
mas_it_gpt = pd.read_csv(
    "/Users/emma/projects/cognoscolab/rassereno/out/formatting/database/all_gpt_mas_it.csv"
)

In [18]:
# --- MAS-IT total vs valence of math: Spearman correlations across groups ---

VALENCE_COLS = [
    "valence_association_1",
    "valence_association_2",
    "valence_association_3",
]

# (label, associations df, mas-it df, participant_id prefix)
GROUPS = [
    ("Psychology students", associations, mas_it, "psychology"),
    ("GPT-OSS psychology", associations_gpt, mas_it_gpt, "gpt_oss_psychology"),
    ("High school students", associations, mas_it, "high_school"),
    ("GPT-OSS high schoolers", associations_gpt, mas_it_gpt, "gpt_oss_high_school"),
]

# two ways of scoring the valence of math for each participant
MEASURES = {
    # mean valence of the 3 associations to 'matematica'
    "mean_of_associations": lambda df: df[VALENCE_COLS].mean(axis=1),
    # valence of the cue word 'matematica' itself
    "math_word_only": lambda df: df["valence_cue_word"],
}


def mas_it_total(mas_df, prefix):
    """Total MAS-IT score per participant for a given group."""
    df = mas_df[mas_df["participant_id"].str.startswith(prefix)]
    mas_cols = [c for c in df.columns if c.startswith("mas_it")]
    return pd.DataFrame(
        {"participant_id": df["participant_id"], "total": df[mas_cols].sum(axis=1)}
    )


def spearman_mas_vs_math(assoc_df, mas_df, prefix, measure):
    print(prefix)
    print(assoc_df["participant_id"].nunique())
    print(len(mas_df))
    """Spearman correlation between MAS-IT total and the chosen math-valence measure."""
    math = assoc_df[
        (assoc_df["participant_id"].str.startswith(prefix))
        & (assoc_df["cue_word"] == "matematica")
    ].copy()
    math["valence"] = MEASURES[measure](math)

    merged = (
        math[["participant_id", "valence"]]
        .merge(mas_it_total(mas_df, prefix), on="participant_id")
        .dropna(subset=["valence", "total"])
    )
    r, p = stats.spearmanr(merged["total"], merged["valence"])
    return {"r": round(r, 3), "p": round(p, 4), "N": len(merged)}

In [19]:
rows = []
for label, assoc_df, mas_df, prefix in GROUPS:
    row = {"group": label}
    for measure in MEASURES:
        res = spearman_mas_vs_math(assoc_df, mas_df, prefix, measure)
        for stat in ("r", "p", "N"):
            row[(measure, stat)] = res[stat]
    rows.append(row)

results = pd.DataFrame(rows).set_index("group")
results.columns = pd.MultiIndex.from_tuples(results.columns, names=["measure", "stat"])
results

psychology
396
396
psychology
396
396
gpt_oss_psychology
388
388
gpt_oss_psychology
388
388
high_school
396
396
high_school
396
396
gpt_oss_high_school
388
388
gpt_oss_high_school
388
388


measure                mean_of_associations              math_word_only  \
stat                                      r       p    N              r   
group                                                                     
Psychology students                  -0.301  0.0000  313         -0.353   
GPT-OSS psychology                   -0.076  0.1794  314         -0.060   
High school students                 -0.514  0.0000   61         -0.501   
GPT-OSS high schoolers               -0.221  0.0843   62         -0.105   

measure                              
stat                         p    N  
group                                
Psychology students     0.0000  308  
GPT-OSS psychology      0.2856  314  
High school students    0.0000   62  
GPT-OSS high schoolers  0.4165   62

In [11]:
def fmt_p(p):
    return "$<$ .001" if p < 0.001 else f"{p:.3f}".lstrip("0")


def results_to_latex(results, measure_labels):
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\caption{Spearman correlations between total MAS-IT and the valence of math.}",
        r"\label{tab:masit_math_valence}",
        r"\begin{tabular}{l" + "rrr" * len(measure_labels) + "}",
        r"\toprule",
        " & "
        + " & ".join(
            rf"\multicolumn{{3}}{{c}}{{{lab}}}" for lab in measure_labels.values()
        )
        + r" \\",
        "Group & " + " & ".join([r"$r$ & $p$ & $N$"] * len(measure_labels)) + r" \\",
        r"\midrule",
    ]
    for group, row in results.iterrows():
        cells = [group]
        for m in measure_labels:
            cells += [
                f"{row[(m,'r')]:.3f}",
                fmt_p(row[(m, "p")]),
                f"{int(row[(m,'N')])}",
            ]
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


measure_labels = {
    "mean_of_associations": "Mean of associations",
    "math_word_only": "Math word only",
}
print(results_to_latex(results, measure_labels))

\begin{table}[ht]
\centering
\caption{Spearman correlations between total MAS-IT and the valence of math.}
\label{tab:masit_math_valence}
\begin{tabular}{lrrrrrr}
\toprule
 & \multicolumn{3}{c}{Mean of associations} & \multicolumn{3}{c}{Math word only} \\
Group & $r$ & $p$ & $N$ & $r$ & $p$ & $N$ \\
\midrule
Psychology students & -0.301 & $<$ .001 & 313 & -0.353 & $<$ .001 & 308 \\
GPT-OSS psychology & -0.076 & .179 & 314 & -0.060 & .286 & 314 \\
High school students & -0.514 & $<$ .001 & 61 & -0.501 & $<$ .001 & 62 \\
GPT-OSS high schoolers & -0.221 & .084 & 62 & -0.105 & .416 & 62 \\
\bottomrule
\end{tabular}
\end{table}


## Concreteness

In [109]:
concreteness_it = pd.read_csv(
    "/Users/emma/projects/cognoscolab/rassereno/data/concreteness_dataset_lemmatized_it.csv"
)

conc_lookup = dict(zip(concreteness_it["lemma"], concreteness_it["Conc.M_mean"]))

In [110]:
assoc_cols = ["association_1", "association_2", "association_3"]

long_df = associations_psy_math.melt(
    id_vars=["participant_id", "cue_word"],
    value_vars=assoc_cols,
    var_name="assoc_slot",
    value_name="response",
)

print("NaNs:", len(long_df[long_df["response"].isna()]))

long_df = long_df.dropna(subset=["response"])
long_df

NaNs: 28


,participant_id,cue_word,assoc_slot,response
0,psychology_001,matematica,association_1,numeri
1,psychology_002,matematica,association_1,fisica
2,psychology_003,matematica,association_1,numeri
3,psychology_004,matematica,association_1,algebra
4,psychology_005,matematica,association_1,voti
...,...,...,...,...
943,psychology_312,matematica,association_3,articolata
944,psychology_313,matematica,association_3,calcoli
945,psychology_314,matematica,association_3,operazioni
946,psychology_315,matematica,association_3,formule


In [111]:
def lemmatize(text, nlp):
    doc = nlp(str(text).strip().lower())
    return " ".join(tok.lemma_ for tok in doc)


long_df["lemma"] = long_df["response"].apply(lambda x: lemmatize(x, nlp_it))
long_df["concreteness"] = long_df["lemma"].map(conc_lookup)

# quick sanity check on match rate
match_rate = long_df["concreteness"].notna().mean()
print(f"Matched {match_rate:.1%} of responses to the concreteness norms")

unmatched = long_df[long_df["concreteness"].isna()]["lemma"].value_counts()
print(unmatched)  # inspect what's falling through (typos, slang, multi-word mismatches)
min_responses = 2  # decide your threshold; you only have 3 responses per participant max, so 2 is reasonable here

participant_conc = (
    long_df.dropna(subset=["concreteness"])
    .groupby("participant_id")
    .agg(
        mean_concreteness=("concreteness", "mean"),
        n_matched=("concreteness", "count"),
    )
    .reset_index()
)

participant_conc = participant_conc[participant_conc["n_matched"] >= min_responses]
print(
    f"Retained {len(participant_conc)} / {associations_psy_math['participant_id'].nunique()} participants"
)
# assuming you have a mas_it_df with columns: participant_id, mas_it_score
merged = participant_conc.merge(mas_it_psy, on="participant_id")

r, p = stats.spearmanr(merged["total"], merged["mean_concreteness"])
print(f"Spearman r = {r:.3f}, p = {p:.4f}, N = {len(merged)}")

Matched 86.2% of responses to the concreteness norms
lemma
formula          27
calcolatrice     11
lavagna           6
derivata          5
teorema           5
                 ..
spressione        1
patriarcato       1
euclide           1
sempre giusto     1
articolato        1
Name: count, Length: 61, dtype: int64
Retained 289 / 316 participants
Spearman r = -0.094, p = 0.1119, N = 289
